# 02 — Export haematoxylin tiles at 224 px / 0.5 µm/px

**Gate this notebook closes: G2.** The heart of the plan, and the only place a mistake is
expensive: every later notebook trusts this output without re-checking it.

The order of operations is not negotiable:

1. **Resample to 0.5 µm/px in intensity space, before the logarithm**, with area averaging.
   A coarser sensor averages the light arriving over a larger area, so averaging
   *transmissions* is what a coarser scan physically is. Averaging densities takes the mean
   of logarithms — the log of a geometric mean — which is biased low and is a number no
   instrument records.
2. **Resample the mask by nearest neighbour.** A label is not an intensity; interpolating one
   invents classes nobody annotated.
3. **White point → optical density → deconvolution**, all through `hchannel`.
4. **Cut the grid, vote, and drop what is too mixed to learn from.**

Everything above lives in `src/export.py`, tested in `tests/test_majority_vote.py`. This
notebook runs it and checks the result.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

In [ ]:
import json
from collections import Counter

import numpy as np
from tqdm.auto import tqdm

import bcss, download_bcss, export, hchannel

spec = export.TileSpec(tile_px=224, mpp=0.5)
print("tile geometry:", json.dumps(spec.descriptor(), indent=2))
print(f"\nfield of view: {spec.tile_um:.0f} um - about nine cells across, which is the")
print("smallest window that can show whether epithelium sits inside a duct or has")


## Export

~1 hour, disk-bound. One region at a time, so progress is visible and a crash costs one
region rather than the run.

In [ ]:
regions = bcss.find_regions(backend_path.BCSS_DIR)

# The measured resolution per slide - NOT the 0.2500 the filenames claim. `source_mpp`
# has no default in the exporter for exactly this reason: a default would be an
# assumption, and this one is wrong by enough to matter.
measured = download_bcss.resolutions(backend_path.BCSS_DIR)
assert measured, ("no slide_mpp.json - run notebook 01's girder download, which "
                  "records each slide's own resolution as it goes")

print(f"{len(regions)} regions to export")
print(f"source resolutions: {sorted({round(measured[r.slide_key], 4) for r in regions})}")
print()

rows: list[dict] = []
drops = Counter()
per_region: list[dict] = []

progress = tqdm(export.iter_regions(regions, backend_path.TILES_DIR, spec=spec,
                                    source_mpp=measured), total=len(regions))
for region, region_rows, region_drops in progress:
    rows.extend(region_rows)
    drops.update(region_drops)
    per_region.append({
        "roi_id": region.roi_id,
        "slide_id": region.slide_id,
        "institution": region.institution,
        "kept": len(region_rows),
        **region_drops,
        # The region's own invasive share, for the tumour-content stratification in
        # notebook 04. A property of the tissue, read off the labels, never off a
        # prediction.
        "invasive_share": (
            float(np.mean([r["label"] == bcss.INVASIVE for r in region_rows]))
            if region_rows else 0.0
        ),
    })
    progress.set_postfix(kept=len(rows))

print()
print(f"kept {len(rows):,} tiles")
print(f"dropped: {dict(drops)}")

In [ ]:
manifest_path = export.write_manifest(
    rows, dict(drops), backend_path.TILES_DIR, spec=spec,
    extra={"per_region": per_region, "regions_exported": len(regions),
           "source_mpp_measured": {r.slide_key: measured[r.slide_key]
                                   for r in regions}},
)
print("manifest:", manifest_path)

summary = json.loads((backend_path.TILES_DIR / "export_summary.json").read_text())
print(json.dumps({k: v for k, v in summary.items() if k != "per_region"}, indent=2))
print(f"per_region: <{len(summary['per_region'])} rows>")

## G2 — the checks, before a single feature is extracted

Four assertions and one thing only a human can do.

In [ ]:
import datasets

rows_read = datasets.read_manifest(manifest_path)
labels = np.array([int(r["label"]) for r in rows_read])

print(f"tiles          : {len(rows_read):,}   (the plan's estimate was ~14,000)")
print(f"slides         : {len({r['slide_id'] for r in rows_read})}")
print(f"institutions   : {len({r['institution'] for r in rows_read})}")
print()
for cls, name in enumerate(bcss.CLASS_NAMES):
    count = int((labels == cls).sum())
    print(f"  {name:<26} {count:>8,}  {count / len(labels):6.1%}")

# The rare class is the informative one. BCSS has plenty of dcis, so if the
# non-invasive class is under ~2% the mapping or the vote is wrong - not the dataset.
non_invasive_share = float((labels == bcss.NON_INVASIVE).mean())
assert non_invasive_share > 0.02, (
    f"non-invasive is only {non_invasive_share:.2%} of tiles. BCSS has plenty of dcis "
    "and normal ducts, so this points at the code mapping or the vote thresholds."
)

# All three classes present, or a three-class head cannot be fitted.
assert set(np.unique(labels)) == {0, 1, 2}, "a class is missing entirely"

# Drops accounted for.
total = len(rows_read) + sum(drops.values())
print(f"\naccounted for   : {len(rows_read):,} kept + {sum(drops.values()):,} dropped "
      f"= {total:,} candidate tiles")

print("\nG2a OK - class balance is plausible and every candidate tile is accounted for")

In [ ]:
# No slide may appear on both sides of any split. Asserted, never eyeballed: a leak
# is invisible in every downstream number except as an implausibly good one, and an
# implausibly good number is the last thing anyone questions.
train, test = datasets.institution_split(rows_read)
datasets.assert_no_leak(train, test)
print(f"institution split: {len(train):,} train / {len(test):,} test tiles")

folds = datasets.slide_folds(train, folds=5)
datasets.assert_no_leak(*[[train[i] for i in fold] for fold in folds])
print(f"slide folds      : {[len(f) for f in folds]}")

weights = datasets.class_weights(train)
print(f"class weights    : {dict(zip(bcss.CLASS_NAMES, [round(float(w), 3) for w in weights]))}")
print("\nG2b OK - no slide crosses a split boundary")

### The eyeball check

Twelve random tiles per class. A non-expert can confirm class `0` (fat, stroma — stringy or
bubbly, few nuclei) and class `2` (dense, disorganised). **If class `2` tiles look like empty
space, the mask is misaligned with the image** — check the resample and any off-by-one crop
before going any further, because no later gate can see that fault.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 12, figsize=(16, 4.4))

for cls, name in enumerate(bcss.CLASS_NAMES):
    pool = [r for r in rows_read if int(r["label"]) == cls]
    picks = rng.choice(len(pool), size=min(12, len(pool)), replace=False)
    for column in range(12):
        ax = axes[cls, column]
        ax.axis("off")
        if column >= len(picks):
            continue
        row = pool[int(picks[column])]
        stored = np.asarray(Image.open(backend_path.TILES_DIR / row["tile_path"]))
        ax.imshow(stored, cmap="bone_r", vmin=0, vmax=255)
        if column == 0:
            ax.set_title(f"{name}  (n={len(pool):,})", fontsize=9, loc="left")

plt.tight_layout()
plt.savefig(backend_path.REPORTS_DIR / "02_tiles_by_class.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# Reproducibility: re-exporting one region must give byte-identical tiles. If it does
# not, something in the path is non-deterministic and the feature cache cannot be
# trusted to match the manifest.
import hashlib

region = regions[0]
again, _ = export.export_region(
    region, spec=spec, source_mpp=summary['source_mpp_measured'][region.slide_key])
first = again[0]

on_disk = np.asarray(Image.open(
    backend_path.TILES_DIR / bcss.CLASS_NAMES[int(first.row["label"])]
    / f"{first.row['tile_id']}.png"
))
assert np.array_equal(first.stored, on_disk), "re-export does not reproduce the tile"
print(f"re-export of {region.roi_id[:28]} reproduces tile 0 exactly")
print(f"  sha256 {hashlib.sha256(first.stored.tobytes()).hexdigest()[:32]}")


---

**G2 closed?** Then the features are worth caching, and BCSS itself can be deleted to
reclaim ~10 GB — the tiles are the training set from here on.

- [ ] tile count within ~2× of the estimate, drops accounted for
- [ ] all three classes present, non-invasive above ~2 %
- [ ] no slide crosses a split boundary (asserted)
- [ ] tiles look like tissue, and class 2 is not empty space
- [ ] re-export is byte-identical